# Texas Flood Losses

In [1]:
import duckdb
import prep as p

con = duckdb.connect()

p.load_claims(con)

con.sql("SELECT COUNT(*) AS tx_rows FROM claims").df()

,tx_rows
0,394204


In [2]:
table = 'data/tx_claims.parquet'

con.sql('''
            SELECT 
                COUNT(*) as total_rows,
                COUNT(DISTINCT id) as unique_ids, 
                COUNT(*) - count(id) as nulls
            FROM
                claims
                
        ''').df()

,total_rows,unique_ids,nulls
0,394204,394204,0


In [3]:
con.sql("DESCRIBE claims").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,dateOfLoss,TIMESTAMP WITH TIME ZONE,YES,None,None,None
2,yearOfLoss,BIGINT,YES,None,None,None
3,state,VARCHAR,YES,None,None,None
4,countyCode,VARCHAR,YES,None,None,None
5,reportedZipCode,VARCHAR,YES,None,None,None
6,netBuildingPaymentAmount,DOUBLE,YES,None,None,None
7,netContentsPaymentAmount,DOUBLE,YES,None,None,None
8,amountPaidOnBuildingClaim,DOUBLE,YES,None,None,None
9,amountPaidOnContentsClaim,DOUBLE,YES,None,None,None


In [4]:
con.sql('''
        select column_name, null_percentage
        from (SUMMARIZE claims)''').df()

,column_name,null_percentage
0,id,0.00
1,dateOfLoss,0.00
2,yearOfLoss,0.00
3,state,0.00
4,countyCode,0.54
5,reportedZipCode,1.30
6,netBuildingPaymentAmount,0.00
7,netContentsPaymentAmount,0.00
8,amountPaidOnBuildingClaim,18.97
9,amountPaidOnContentsClaim,18.97


In [5]:

''' Paid claim definition: a claim counts as paid when netBuildingPaymentAmount or netContentsPaymentAmount is above zero.
    Check: every claim should either have a positive net payment or a reason code
    in nonPaymentReasonBuilding or nonPaymentReasonContents.
    Result: true for nearly all claims. Only 950 rows (about 0.2%) have no reason
    code and no positive payment.
    Note: the amountPaidOnBuildingClaim and amountPaidOnContentsClaim columns are
    not safe for this check. Null values fail every comparison in a CASE, so they
    land in the ELSE branch, which labeled about 68,000 unpaid claims as paid.
'''

con.sql(''' 
        select 
            case 
                when 
                    nonPaymentReasonBuilding is not null
                    or
                    nonPaymentReasonContents is not null
                    or
                    netBuildingPaymentAmount > 0 
                    or
                    netContentsPaymentAmount > 0
                    then 'Explained'
                    else 'Not Explained'
                    end as cases_explained,
             count(*) as "number_of_claims"
        from claims
        group by cases_explained
        ''').df()

,cases_explained,number_of_claims
0,Not Explained,950
1,Explained,393254


In [6]:
con.sql('''
            select 
                count(*) as entry_errors,
            from claims
            where nonPaymentReasonBuilding = '98'
                  or
                  nonPaymentReasonContents = '98'
''').df()

,entry_errors
0,5867


In [7]:
con.sql('''
            select 
                case when nonPaymentReasonBuilding = '98' then 'error' else 'good' end as 'building_code',
                case when netContentsPaymentAmount > 0 then 'payment' else 'no_payment' end as 'contents_payments',
                count(*) as 'count'
            from claims
            where nonPaymentReasonBuilding = '98'
                  or
                  nonPaymentReasonContents = '98'
            group by building_code, contents_payments 
''').df()

,building_code,contents_payments,count
0,good,payment,3
1,error,no_payment,4737
2,good,no_payment,994
3,error,payment,133


In [30]:
con.sql('''
            select 
                case when nonPaymentReasonContents = '98' then 'error' else 'good' end as contents_code,
                case when netBuildingPaymentAmount > 0 then 'payment' else 'no_payment' end as building_payments,
                count(*) as case_count
            from claims
            where nonPaymentReasonBuilding = '98'
                  or
                  nonPaymentReasonContents = '98'
            group by contents_code, building_payments
''').df()

ParserException: Parser Error: syntax error at or near "null"

LINE 3: ...               case when nonPaymentReasonContents = '98' or is null then 'error' else 'good' end as contents_code,
                                                                          ^

In [33]:
con.sql('''
        
            select 
                count(case when nonPaymentReasonContents = '98' or nonPaymentReasonBuilding = '98' then 1 end) as count,
                sum(case when nonPaymentReasonContents = '98' then netBuildingPaymentAmount end) as total_building_paid,
                round(sum(case when nonPaymentReasonContents = '98' then netBuildingPaymentAmount end) / sum(netBuildingPaymentAmount) * 100,3) as percent_building_paid,
                sum(case when nonPaymentReasonBuilding = '98' then netContentsPaymentAmount end) as total_contents_paid,
                round(sum(case when nonPaymentReasonBuilding = '98' then netContentsPaymentAmount end) / sum(netContentsPaymentAmount) * 100,3) as percent_contents_paid
            from claims;
            
''').df()

,count,total_building_paid,percent_building_paid,total_contents_paid,percent_contents_paid
0,5867,9367774.93,0.07,1056746.1,0.027


Dropping 5,867 rows 1.5% of data due to being marked as errors in either contents or building reason codes

In [10]:
con.sql('''
            select 
               count(*)
            from claims
            where nonPaymentReasonContents = '98' or 
                  nonPaymentReasonBuilding = '98'
            
            
''').df()

,count_star()
0,5867


-- countyCode: 2,128 rows are null and 1 row has a non-Texas code (22029, Louisiana).
-- All of these rows are kept, since the state field marks them as Texas claims.
-- The clean sets the non-Texas code to null, so these rows count in Texas totals
-- but not in any county.

In [11]:
con.sql('''
            select 
               count(*) as total,
               count(*) - sum(case when countyCode like '48%' then 1 else 0 end) as non_texas,
               sum(case when countyCode is null then 1 else 0 end) as nulls
            from claims
''').df()

,total,non_texas,nulls
0,394204,2129.0,2128.0


In [34]:
con.sql('''
            select 
               count(*) - sum(case when reportedZipCode like '75%'  or
                                        reportedZipCode like '76%'  or
                                        reportedZipCode like '77%'  or
                                        reportedZipCode like '78%'  or
                                        reportedZipCode like '79%'  or
                                        reportedZipCode like '733%' or
                                        reportedZipCode like '885%'  
                                        then 1 else 0 end) as non_texas,
               sum(case when reportedZipCode is null then 1 else 0 end) as nulls,
               non_texas - nulls as non_texas_non_nulls
            from claims
''').df()

,non_texas,nulls,non_texas_non_nulls
0,5125.0,5116.0,9.0


In [35]:
con.sql('''
            select 
                    reportedZipCode, 
                    count(*) as non_texas_counts
                    
            from 
                    claims
            where   
                    reportedZipCode not  like '75%'  and
                    reportedZipCode not  like '76%'  and
                    reportedZipCode not  like '77%'  and
                    reportedZipCode not  like '78%'  and
                    reportedZipCode not  like '79%'  and
                    reportedZipCode not  like '733%' and
                    reportedZipCode not  like '885%' and
            group by 
                    reportedZipCode
            
''').df()

,reportedZipCode,non_texas_counts
0,11111,1
1,72330,1
2,73070,1
3,70377,1
4,39169,1
5,70000,2
6,96799,2


-- reportedZipCode: 5,116 rows are null and 9 rows have non-Texas zips
-- (mostly neighboring states, plus placeholders like 11111).
-- All of these rows are kept, since the state field marks them as Texas claims.
-- The clean sets the non-Texas zips to null, so these rows count in Texas totals
-- but are left out of zip-level results.

In [14]:
con.sql('''
            select 
                   netBuildingPaymentAmount + netContentsPaymentAmount as total_payment_amount,
                   occupancyType as building_type
            from claims
            order by 
                   total_payment_amount
            limit 10
''').df()

,total_payment_amount,building_type
0,-154798.48,1
1,-133613.83,13
2,-123645.23,11
3,-57688.01,11
4,-35125.37,11
5,-33415.36,3
6,-26557.38,11
7,-25192.95,11
8,-25000.00,1
9,-13529.73,11


In [15]:
con.sql('''
            select 
                       id,
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) as per_unit_building_paid,
                       coalesce(netContentsPaymentAmount,0) as contents_paid,
                       occupancyType as building_type,
                       policyCount,
                       numberOfUnits
            from 
                       claims
            where 
                       per_unit_building_paid > 250000
            order by 
                        coalesce(netBuildingPaymentAmount,0) + coalesce(netContentsPaymentAmount,0) desc
            limit 50
''').df()

,id,per_unit_building_paid,contents_paid,building_type,policyCount,numberOfUnits
0,8138362,500000.0,500000.0,6,1,1
1,6609783,500000.0,500000.0,6,1,1
2,7856582,500000.0,500000.0,4,1,1
3,7238190,500000.0,500000.0,18,1,1
4,7360208,500000.0,500000.0,4,1,1
5,7369400,500000.0,500000.0,4,1,1
6,7406570,500000.0,500000.0,18,1,1
7,7427044,500000.0,500000.0,6,1,1
8,7212552,500000.0,500000.0,6,1,1
9,7470161,500000.0,500000.0,4,1,1


In [16]:
con.sql('''
            select 
                       occupancyType as building_type,
                       count(*)
            from 
                       claims
            where 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) > 250000
            group by   
                       occupancyType

''').df()

,building_type,count_star()
0,1,5
1,2,15
2,3,257
3,4,399
4,6,337
5,11,5
6,12,1
7,13,396
8,18,489


In [17]:
con.sql('''
            select 
                       id, 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) as per_unit_payment, 
                       policyCount,
                       dateOfLoss                      
            from 
                       claims
            where 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) > 250000
                       and occupancyType in (1,2,11,12)
            order by 
                       dateOfLoss desc

''').df()

,id,per_unit_payment,policyCount,dateOfLoss
0,6711554,373911.88,1,2019-09-19 00:00:00+00:00
1,7270906,314648.43,1,2019-09-19 00:00:00+00:00
2,6052790,309173.34,1,2019-09-19 00:00:00+00:00
3,7135170,329506.86,1,2019-09-19 00:00:00+00:00
4,6748066,293522.49,1,2019-09-19 00:00:00+00:00
5,6395078,286240.64,1,2019-09-19 00:00:00+00:00
6,6061297,342354.53,1,2017-08-29 00:00:00+00:00
7,7718106,464451.64,1,2017-08-29 00:00:00+00:00
8,6411264,319849.85,1,2017-08-27 00:00:00+00:00
9,6969681,363997.94,1,2017-08-27 00:00:00+00:00


In [18]:
con.sql('''
            select 
                sum(case
                         when coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) > 250000 and occupancyType in (1,2,11,12)
                         then coalesce(netBuildingPaymentAmount,0) + coalesce(netContentsPaymentAmount,0)
                         else 0 end) as suspect_payments,
                         
                         sum(coalesce(netBuildingPaymentAmount,0) + coalesce(netContentsPaymentAmount,0)) as total_payments,

                         round(suspect_payments / total_payments * 100,2) as percent_suspect_payments
                from 
                         claims

        ''').df()

,suspect_payments,total_payments,percent_suspect_payments
0,9466816.09,1.734376e+10,0.05


26 single-family and 2 to 4 family claims exceed the $250,000 building cap, clustered in Ike, Harvey and Imelda; they’re kept and flagged, and they account for 0.05% of net dollars paid.

In [19]:
con.sql('''
            select 
                       id, 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) as per_unit_payment, 
                       policyCount,
                       dateOfLoss                      
            from 
                       claims
            where 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) > 250000
                       and occupancyType in (1,2,11,12)
            order by 
                       dateOfLoss desc

''').df()

,id,per_unit_payment,policyCount,dateOfLoss
0,6711554,373911.88,1,2019-09-19 00:00:00+00:00
1,7270906,314648.43,1,2019-09-19 00:00:00+00:00
2,6052790,309173.34,1,2019-09-19 00:00:00+00:00
3,7135170,329506.86,1,2019-09-19 00:00:00+00:00
4,6748066,293522.49,1,2019-09-19 00:00:00+00:00
5,6395078,286240.64,1,2019-09-19 00:00:00+00:00
6,6061297,342354.53,1,2017-08-29 00:00:00+00:00
7,7718106,464451.64,1,2017-08-29 00:00:00+00:00
8,6411264,319849.85,1,2017-08-27 00:00:00+00:00
9,6969681,363997.94,1,2017-08-27 00:00:00+00:00


In [20]:
con.sql('''
            select 
                       id, 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) as per_unit_payment, 
                       policyCount,
                       dateOfLoss                      
            from 
                       claims
            where 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) > 500000
                       and occupancyType in (3,13)
            order by 
                       dateOfLoss desc

''').df()

,id,per_unit_payment,policyCount,dateOfLoss


In [21]:
con.sql('''
            select 
                       id, 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) as per_unit_payment, 
                       policyCount,
                       dateOfLoss                      
            from 
                       claims
            where 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) > 250_000
                       and occupancyType = 3
                       and condominiumCoverageTypeCode in ('H','L','A')
            order by 
                       dateOfLoss desc

''').df()

,id,per_unit_payment,policyCount,dateOfLoss


-- Big payments check: I tested whether the largest net payments fit under
-- the NFIP coverage cap for their occupancy type. Building payments are
-- divided by policyCount so condo association policies (one row covering
-- many units) are compared per unit.
--
-- Checks out:
--   Non-residential (4, 6, 18): top claims stop at exactly $500,000
--   building plus $500,000 contents, the cap. None over.
--   Condo associations (code 3, condominiumCoverageTypeCode A, H or L):
--   none over $250,000 per unit.
--   Other residential, 5+ units (3 and 13): none over $500,000 per policy.
--   [Code 3 non-condo before June 2014, old $250,000 cap: RESULT]
--
-- Suspect:
--   Single family and 2 to 4 family (1, 2, 11, 12): 26 claims exceed the
--   $250,000 building cap, up to about 2.2 times. All come from Hurricane
--   Ike (2008), Hurricane Harvey (2017) and Tropical Storm Imelda (2019).
--   They total about $9.5 million, 0.05% of net dollars paid. The cause
--   can't be identified from this data.
--
-- Decision: all large payments stay in. The 26 suspect claims are kept and
-- flagged as a limitation; they're too small a share to change any finding.
-- Side note: numberOfUnits is only filled for condo master policies (0 on
-- other rows), so policyCount is the unit count used throughout.

In [22]:
con.sql('''
            select 
                       id, 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) as per_unit_payment, 
                       policyCount,
                       dateOfLoss                      
            from 
                       claims
            where 
                       coalesce(netBuildingPaymentAmount,0) / nullif(policyCount,0) > 250_000
                       and occupancyType = 3
                       and coalesce(condominiumCoverageTypeCode, 'none') not in ('H','L','A')
                       and dateOfLoss < DATE '2014-06-01'
            order by 
                       dateOfLoss desc

''').df()

,id,per_unit_payment,policyCount,dateOfLoss


--   Other residential, non-condo (code 3) before June 2014, old $250,000
--   cap: none over. All code 3 claims above $250,000 come after the cap
--   rose to $500,000 in June 2014.

In [23]:
con.sql('''
            select 
                       originalConstructionDate,
                       count(*) as count
            from 
                       claims
            where 
                       originalConstructionDate = '1492-10-12'
            group by 
                       originalConstructionDate
            order by 
                       originalConstructionDate

''').df()

,originalConstructionDate,count
0,1492-10-12 00:00:00+00:00,53495


In [24]:
con.sql('''
            select 
                       min(originalConstructionDate) as min_con_date,
                       count(*) as count
            from 
                       claims
            where
                       originalConstructionDate <> '1492-10-12'
            group by  
                       originalConstructionDate
            order by
                       originalConstructionDate asc
            limit 50
            
''').df()

,min_con_date,count
0,1850-01-01 00:00:00+00:00,1
1,1863-01-01 00:00:00+00:00,1
2,1870-01-01 00:00:00+00:00,1
3,1873-01-01 00:00:00+00:00,1
4,1874-01-01 00:00:00+00:00,1
5,1887-07-01 00:00:00+00:00,1
6,1889-01-01 00:00:00+00:00,1
7,1890-01-01 00:00:00+00:00,2
8,1892-01-01 00:00:00+00:00,1
9,1894-01-01 00:00:00+00:00,1


In [25]:
con.sql('''
            select 
                       originalConstructionDate as min_con_date,
                       count(*) as count
            from 
                       claims
            where
                       originalConstructionDate <> '1492-10-12'
            group by  
                       originalConstructionDate
            order by
                       count desc
            limit 50
            
''').df()

,min_con_date,count
0,1970-01-01 00:00:00+00:00,7235
1,1974-12-31 00:00:00+00:00,6848
2,1974-07-01 00:00:00+00:00,6661
3,1978-01-01 00:00:00+00:00,6195
4,1980-01-01 00:00:00+00:00,5552
5,1975-01-01 00:00:00+00:00,5375
6,1960-01-01 00:00:00+00:00,5368
7,1965-01-01 00:00:00+00:00,5332
8,1972-01-01 00:00:00+00:00,4815
9,1979-01-01 00:00:00+00:00,4691


In [37]:
con.sql('''select 
        count(*) as total,
        count(*) - sum(case when countyCode like '48%' then 1 else 0 end) as non_texas,
        sum(case when countyCode is null then 1 else 0 end) as nulls,
        non_texas - nulls as non_texas_non_nulls
from 
        claims
''').df()

,total,non_texas,nulls,non_texas_non_nulls
0,394204,2129.0,2128.0,1.0
